In [5]:
import numpy as np
import pandas as pd

from itertools import combinations, permutations
from operator import itemgetter

In [6]:
def trans_df2dict(df):
    """将数据转化成字典格式"""
    user_rating = dict()  # 用户评分数据
    for row in df.values:
        user_id, movie_id, rating = row[0], row[1], row[2]
        if user_id not in user_rating.keys():
            user_rating[user_id] = {}
        user_rating[user_id][movie_id] = rating
    return user_rating

In [7]:
def get_items_similarity(df, item_num):
    """计算items相似性矩阵，返回相似性矩阵"""
    # 1. 建立用户-物品的倒排表
    inverted_table = df.groupby(by='userid')['moveid'].agg(list).to_dict()

    # 2. 初始化共现矩阵，遍历每个用户，将物品两两组合，并在共现矩阵中加1
    W = np.zeros((item_num, item_num))

    # 统计每个电影被多少人看过
    count_item_users_num = df.groupby(by='moveid')['userid'].agg('count').to_dict()

    for key, val in inverted_table.items():
        val.sort(reverse=True)  # 降序
        for per in combinations(val, 2):
            W[per[0] - 1][per[1] - 1] += 1
            W[per[1] - 1][per[0] - 1] += 1

    # 计算相似性
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            W[i][j] /= np.sqrt(count_item_users_num.get(i + 1) * count_item_users_num.get(j + 1))

    w_dict = {}
    for i in range(W.shape[0]):
        tmp = []
        for index, k in enumerate(W[i]):
            tmp.append((index + 1, k))
        w_dict[i + 1] = tmp
    return w_dict

In [8]:
def user_interest_with_items(user_id, item_id, K, user_rating, w_dict):
    """计算指定用户与指定物品的兴趣程度"""
    interest = 0
    for i in sorted(w_dict[item_id], key=itemgetter(1), reverse=True)[0:K]:
        item_index = i[0]
        item_simi = i[1]
        if item_index in user_rating[user_id].keys():
            interest += item_simi * user_rating[user_id][item_index]
    return interest

In [9]:
def get_user_interest_list(user_id, K, user_rating, w_dict):
    """计算用户的兴趣列表"""
    rank = []
    item_id_list = w_dict.keys()
    for item_id in item_id_list:
        if item_id in user_rating[user_id].keys():
            continue
        interest = user_interest_with_items(user_id, item_id, K, user_rating, w_dict)
        rank.append((item_id, interest))
    return sorted(rank, key=itemgetter(1), reverse=True)


In [15]:
df = pd.read_csv(r'D:\PythonEx\machinelearningIntro\机器学习实践\关联规则\data\ratings.csv').head(100)

In [16]:
user_rating = trans_df2dict(df)

In [17]:
inverted_table = df.groupby(by='userid')['moveid'].agg(list).to_dict()

In [18]:
item_num = df.moveid.nunique()

In [19]:
w_dict = get_items_similarity(df, item_num)
w_dict

IndexError: index 3407 is out of bounds for axis 0 with size 99

In [19]:
if  __name__ == '__main__':
    df = pd.read_csv('D:/Download_D/ratings.csv')
    item_num = df.moveid.nunique()
    user_num = df.userid.nunique()
   # print(item_num)
   # print(user_num)
    user_rating = trans_df2dict(df)
    
    w_dict = get_items_similarity(df, item_num)
    
    recommend_list = get_user_interest_list(2, 20, user_rating, w_dict)
    print(recommend_list[0:20])

IndexError: index 3892 is out of bounds for axis 0 with size 3706

In [ ]:
#!/usr/bin/python3
# -*- coding: utf-8 -*-
from numpy import *
import time
from texttable import Texttable
class CF:
  def __init__(self, ratings, k=5, n=10):
    self.ratings = ratings
    # 邻居个数
    self.k = k
    # 推荐个数
    self.n = n
    # 用户对电影的评分
    # 数据格式{'UserID：用户ID':[(MovieID：电影ID,Rating：用户对电影的评星)]}
    self.userDict = {}
    # 对某电影评分的用户
    # 数据格式：{'MovieID：电影ID',[UserID：用户ID]}
    # {'1',[1,2,3..],...}
    self.ItemUser = {}
    # 邻居的信息
    self.neighbors = []
    # 推荐列表
    self.recommandList = []
    self.cost = 0.0

  # 基于用户的推荐
  # 根据对电影的评分计算用户之间的相似度
  def recommendByUser(self, userId):
    self.formatRate()
    # 推荐个数 等于 本身评分电影个数，用户计算准确率
    self.n = len(self.userDict[userId])
    self.getNearestNeighbor(userId)
    self.getrecommandList(userId)
    self.getPrecision(userId)

  # 获取推荐列表
  def getrecommandList(self, userId):
    self.recommandList = []
    # 建立推荐字典
    recommandDict = {}
    for neighbor in self.neighbors:
      movies = self.userDict[neighbor[1]]
      for movie in movies:
        if(movie[0] in recommandDict):
          recommandDict[movie[0]] += neighbor[0]
        else:
          recommandDict[movie[0]] = neighbor[0]

    # 建立推荐列表
    for key in recommandDict:
      self.recommandList.append([recommandDict[key], key])
    self.recommandList.sort(reverse=True)
    self.recommandList = self.recommandList[:self.n]

  # 将ratings转换为userDict和ItemUser
  def formatRate(self):
    self.userDict = {}
    self.ItemUser = {}
    for i in self.ratings:
      # 评分最高为5 除以5 进行数据归一化
      temp = (i[1], float(i[2]) / 5)
      # 计算userDict {'1':[(1,5),(2,5)...],'2':[...]...}
      if(i[0] in self.userDict):
        self.userDict[i[0]].append(temp)
      else:
        self.userDict[i[0]] = [temp]
      # 计算ItemUser {'1',[1,2,3..],...}
      if(i[1] in self.ItemUser):
        self.ItemUser[i[1]].append(i[0])
      else:
        self.ItemUser[i[1]] = [i[0]]

  # 找到某用户的相邻用户
  def getNearestNeighbor(self, userId):
    neighbors = []
    self.neighbors = []
    # 获取userId评分的电影都有那些用户也评过分
    for i in self.userDict[userId]:
      for j in self.ItemUser[i[0]]:
        if(j != userId and j not in neighbors):
          neighbors.append(j)
    # 计算这些用户与userId的相似度并排序
    for i in neighbors:
      dist = self.getCost(userId, i)
      self.neighbors.append([dist, i])
    # 排序默认是升序，reverse=True表示降序
    self.neighbors.sort(reverse=True)
    self.neighbors = self.neighbors[:self.k]

  # 格式化userDict数据
  def formatuserDict(self, userId, l):
    user = {}
    for i in self.userDict[userId]:
      user[i[0]] = [i[1], 0]
    for j in self.userDict[l]:
      if(j[0] not in user):
        user[j[0]] = [0, j[1]]
      else:
        user[j[0]][1] = j[1]
    return user

  # 计算余弦距离
  def getCost(self, userId, l):
    # 获取用户userId和l评分电影的并集
    # {'电影ID'：[userId的评分，l的评分]} 没有评分为0
    user = self.formatuserDict(userId, l)
    x = 0.0
    y = 0.0
    z = 0.0
    for k, v in user.items():
      x += float(v[0]) * float(v[0])
      y += float(v[1]) * float(v[1])
      z += float(v[0]) * float(v[1])
    if(z == 0.0):
      return 0
    return z / sqrt(x * y)

  # 推荐的准确率
  def getPrecision(self, userId):
    user = [i[0] for i in self.userDict[userId]]
    recommand = [i[1] for i in self.recommandList]
    count = 0.0
    if(len(user) >= len(recommand)):
      for i in recommand:
        if(i in user):
          count += 1.0
      self.cost = count / len(recommand)
    else:
      for i in user:
        if(i in recommand):
          count += 1.0
      self.cost = count / len(user)

  # 显示推荐列表
  def showTable(self):
    neighbors_id = [i[1] for i in self.neighbors]
    table = Texttable()
    table.set_deco(Texttable.HEADER)
    table.set_cols_dtype(["t", "t", "t", "t"])
    table.set_cols_align(["l", "l", "l", "l"])
    rows = []
    rows.append([u"movie ID", u"Name", u"release", u"from userID"])
    for item in self.recommandList:
      fromID = []
      for i in self.movies:
        if i[0] == item[1]:
          movie = i
          break
      for i in self.ItemUser[item[1]]:
        if i in neighbors_id:
          fromID.append(i)
      movie.append(fromID)
      rows.append(movie)
    table.add_rows(rows)
    print(table.draw())
# 获取数据
def readFile(filename):
  #files = open(filename, "r", encoding="utf-8")
  # 如果读取不成功试一下
  files = open(filename, "r", encoding="iso-8859-15")
  data = []
  for line in files.readlines():
    item = line.strip().split("::")
    data.append(item)
  return data

# -------------------------开始-------------------------------
start = time.clock()
movies = readFile("E:/数据挖掘整理/ml-1m/movies.dat")
ratings = readFile("E:/数据挖掘整理/ml-1m/ratings.dat")
demo = CF(movies, ratings, k=20)
demo.recommendByUser("100")
print("推荐列表为：")
demo.showTable()
print("处理的数据为%d条" % (len(demo.ratings)))
print("准确率： %.2f %%" % (demo.cost * 100))
end = time.clock()
print("耗费时间： %f s" % (end - start))
